# Animal Kingdom species and suitability review

This notebook authenticates to the private bucket, streams the action-video archive once, extracts only the deterministic review sample, and saves labels back to the bucket. It does not use a GPU or train a model.

Run each cell in order. The extraction cell reads the complete compressed 15.6 GB archive but stores only approximately 203 selected videos on the temporary Colab disk. Do not disconnect while it runs.

In [ ]:
from google.colab import auth

auth.authenticate_user()

PROJECT_ID = 'cat-behaviour-research'
BUCKET = 'cat-behaviour-research-raluca-2026'
REPO_URL = 'https://github.com/ralucabiras/explainable-cat-behaviour-interpreter.git'
REPO_DIR = '/content/explainable-cat-behaviour-interpreter'
PLAN_PATH = f'{REPO_DIR}/backend/video_dataset/review/animal-kingdom-review-v1.plan.json'
LABELS_URI = f'gs://{BUCKET}/curated/review-v1/review-labels.json'

!gcloud config set project {PROJECT_ID}
!test -d {REPO_DIR}/.git || git clone {REPO_URL} {REPO_DIR}
print('Authentication and repository setup complete.')

In [ ]:
import json
import subprocess
from pathlib import Path

plan = json.loads(Path(PLAN_PATH).read_text())
extract_root = Path('/content/animal-kingdom-review')
extract_root.mkdir(parents=True, exist_ok=True)
members_path = extract_root / 'members.txt'
members = sorted({item['archive_member'] for item in plan['items']})
members_path.write_text('\n'.join(members) + '\n')
print(f"Plan: {plan['plan_version']}")
print(f'Selected source videos: {len(members)}')
print(f"Archive: {plan['source_archive_uri']}")

In [ ]:
# This streams the archive once and can take a while. It does not save the full archive.
gcloud = subprocess.Popen(
    ['gcloud', 'storage', 'cat', plan['source_archive_uri']],
    stdout=subprocess.PIPE,
)
tar = subprocess.run(
    ['tar', '-xzf', '-', '-C', str(extract_root), '-T', str(members_path)],
    stdin=gcloud.stdout,
)
if gcloud.stdout:
    gcloud.stdout.close()
gcloud_code = gcloud.wait()
if tar.returncode or gcloud_code:
    raise RuntimeError(f'Extraction failed: gcloud={gcloud_code}, tar={tar.returncode}')
missing = [member for member in members if not (extract_root / member).is_file()]
if missing:
    raise RuntimeError(f'{len(missing)} planned archive members were missing; first: {missing[:5]}')
print(f'Extracted and verified {len(members)} review videos.')

## Review labels

For each video, select the visible species and whether the clip is suitable for learning the listed observable actions. Use `unclear` instead of guessing. A checkpoint is uploaded every five saved reviews and when **Checkpoint now** is pressed.

In [ ]:
import ipywidgets as widgets
from IPython.display import Video, clear_output, display

labels_path = extract_root / 'review-labels.json'
download = subprocess.run(
    ['gcloud', 'storage', 'cp', LABELS_URI, str(labels_path)],
    capture_output=True,
)
if download.returncode == 0:
    saved = json.loads(labels_path.read_text())
    labels = {item['item_id']: item for item in saved.get('labels', [])}
    print(f'Resuming with {len(labels)} saved labels.')
else:
    labels = {}
    print('Starting a new review.')

state = {'index': 0, 'saves_since_checkpoint': 0}
species_options = [
    'unreviewed', 'domestic_cat', 'wild_feline',
    'other_mammal', 'non_mammal', 'unclear',
]
suitability_options = ['unreviewed', 'suitable', 'unsuitable', 'unclear']
action_options = sorted(plan['coverage_by_action'])
output = widgets.Output()

def checkpoint():
    payload = {
        'schema_version': '1.0.0',
        'plan_version': plan['plan_version'],
        'labels': [labels[key] for key in sorted(labels)],
    }
    labels_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n')
    result = subprocess.run(['gcloud', 'storage', 'cp', str(labels_path), LABELS_URI])
    if result.returncode:
        raise RuntimeError('Could not upload the review checkpoint')
    state['saves_since_checkpoint'] = 0

def render():
    with output:
        clear_output(wait=True)
        item = plan['items'][state['index']]
        previous = labels.get(item['id'], {})
        print(f"Review {state['index'] + 1}/{len(plan['items'])}: {item['source_clip_id']}")
        print('Mapped actions:', ', '.join(item['actions']))
        display(Video(str(extract_root / item['archive_member']), embed=True, width=640))
        species = widgets.Dropdown(
            options=species_options,
            value=previous.get('species', 'unreviewed'),
            description='Species:',
        )
        suitability = widgets.Dropdown(
            options=suitability_options,
            value=previous.get('suitability', 'unreviewed'),
            description='Suitable:',
        )
        visible = widgets.SelectMultiple(
            options=action_options, value=tuple(previous.get('visible_actions', item['actions'])),
            description='Visible:', rows=7,
        )
        notes = widgets.Textarea(value=previous.get('notes', ''), description='Notes:')
        back = widgets.Button(description='Previous')
        save = widgets.Button(description='Save & next', button_style='success')
        upload = widgets.Button(description='Checkpoint now')

        def go_back(_):
            state['index'] = max(0, state['index'] - 1)
            render()

        def save_next(_):
            if species.value == 'unreviewed' or suitability.value == 'unreviewed':
                print('Choose both species and suitability before saving.')
                return
            labels[item['id']] = {
                'item_id': item['id'], 'species': species.value,
                'suitability': suitability.value, 'visible_actions': list(visible.value),
                'notes': notes.value.strip(),
            }
            state['saves_since_checkpoint'] += 1
            if state['saves_since_checkpoint'] >= 5 or state['index'] == len(plan['items']) - 1:
                checkpoint()
            state['index'] = min(len(plan['items']) - 1, state['index'] + 1)
            render()

        def upload_now(_):
            checkpoint()
            print(f'Checkpoint uploaded with {len(labels)} labels.')

        back.on_click(go_back)
        save.on_click(save_next)
        upload.on_click(upload_now)
        display(species, suitability, visible, notes, widgets.HBox([back, save, upload]))
        print(f'Saved: {len(labels)}/{len(plan["items"])}')

display(output)
render()

In [ ]:
from collections import Counter

# Run after finishing or whenever you want a progress summary.
checkpoint()
print('Reviewed:', len(labels), '/', len(plan['items']))
print('Species:', dict(Counter(item['species'] for item in labels.values())))
print('Suitability:', dict(Counter(item['suitability'] for item in labels.values())))
print('Saved privately to:', LABELS_URI)